# Municipal Complaint Routing: A Comparative Benchmark of Classical ML vs. Transformer Models on Real NYC 311 Data

**Kaggle GPU version.** This notebook is built to run on Kaggle Notebooks with a free GPU accelerator
(Settings → Accelerator → GPU T4 x2 or P100).

### Task design (read this before running)

We predict the **responding `Agency`** (e.g. NYPD, DSNY, DOT, HPD) from the **`Descriptor`** free-text field.

We deliberately do **not** use `Complaint Type` as input text when predicting `Complaint Type` (or vice versa),
because those two columns are two views of the same categorical label assigned by the same operator at intake —
using one to predict the other is close to circular and inflates accuracy artificially.

Predicting `Agency` from `Descriptor` is the genuine, practically useful task: *given what a citizen described,
which department should this be routed to?*

### What this notebook does
1. Loads a NYC 311 CSV added via Kaggle's **Add Data** panel (no Drive, no downloads).
2. Builds a clean `Descriptor -> Agency` classification dataset, capped to the top-N most frequent agencies.
3. Trains and evaluates 4 models on an identical train/test split:
   - TF-IDF + Linear SVM (baseline)
   - TF-IDF + Multinomial Naive Bayes (baseline)
   - BERT-base-uncased (transformer)
   - DeBERTa-v3-small (transformer)
4. Reports macro/micro F1, per-class precision/recall/F1, and confusion matrices for every model.
5. Produces a single comparison table + chart you can drop straight into the paper's Results section.

> Tip: RoBERTa-base can be added by appending its name to `TRANSFORMER_MODELS` below — omitted by default
> to keep total Kaggle GPU runtime reasonable. See the "Add more models" note near that list.


## 1. Setup

In [ ]:
# Kaggle images usually ship recent versions already, but pin/upgrade the essentials.
!pip install -q -U transformers datasets accelerate evaluate scikit-learn statsmodels


In [ ]:
import os, re, json, glob, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from datasets import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2 (or P100) and restart the session.")


## 2. Load the dataset (Kaggle "Add Data" only)

1. Click **Add Data** (top right of the notebook editor).
2. Search for the NYC 311 dataset you want to use (e.g. search "311 service requests") and add it,
   OR upload your own CSV export as a private Kaggle dataset and add that.
3. Run the cell below — it auto-discovers the CSV under `/kaggle/input/` so you don't need to hardcode a path.
   If you have more than one candidate file, it will print all of them and you can set `CSV_PATH` manually.


In [ ]:
candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print(f"Found {len(candidates)} CSV file(s) under /kaggle/input:")
for c in candidates:
    size_mb = os.path.getsize(c) / (1024 * 1024)
    print(f"  {c}  ({size_mb:.1f} MB)")

# 311 dataset bundles often include multiple CSVs (service catalogs, metadata, lookup
# tables) alongside the real complaint-records file. The real one is (a) NOT named
# like a "web-content"/"services" catalog, and (b) by far the largest file, since it's
# millions of individual complaint tickets. Prefer the largest CSV that doesn't look
# like a services catalog.
def looks_like_catalog(path):
    name = os.path.basename(path).lower()
    return "web-content" in name or "services" in name or "dictionary" in name

real_candidates = [c for c in candidates if not looks_like_catalog(c)] or candidates
CSV_PATH = max(real_candidates, key=os.path.getsize) if real_candidates else None
print("\nAuto-selected:", CSV_PATH)
print("If this looks wrong, just set CSV_PATH manually to the correct path from the list above.")


In [ ]:
# The real 311 complaint-records file can be 10+ GB with tens of millions of rows,
# ordered chronologically. Reading just the first N rows (nrows=...) would silently
# give you only the OLDEST years -- a biased, non-representative sample (older complaint
# taxonomy, fewer digitally-reported categories). Instead we:
#   1. Peek the header only (cheap) to find the columns we actually need.
#   2. Stream the file in chunks, loading ONLY those columns (keeps memory low even on
#      a 14GB file), and randomly keep a fraction of rows from every chunk -- giving a
#      sample spread across the entire date range instead of just the earliest rows.

SAMPLE_FRACTION = 0.05   # keep ~5% of rows from every chunk
MAX_SAMPLED_ROWS = 800_000
CHUNK_SIZE = 250_000

header_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
print("Columns found in file header:")
print(header_cols)


In [ ]:
def normalize_col(c):
    c = unicodedata.normalize("NFKD", c)
    c = c.encode("ascii", "ignore").decode("ascii")
    c = c.strip().lower()
    c = " ".join(c.split())
    return c

normalized_header = {normalize_col(c): c for c in header_cols}

def find_original_col(keywords):
    for norm_name, orig_name in normalized_header.items():
        for kw in keywords:
            if kw in norm_name:
                return orig_name
    return None

descriptor_orig = find_original_col(["descriptor"])
agency_orig     = find_original_col(["agency name", "agency"])
complaint_orig  = find_original_col(["complaint type"])

print("Descriptor column:", descriptor_orig)
print("Agency column:", agency_orig)
print("Complaint Type column (reference only):", complaint_orig)

assert descriptor_orig is not None and agency_orig is not None, \
    "Could not auto-detect Descriptor/Agency columns in the header -- inspect header_cols above and set them manually."

usecols = [c for c in [descriptor_orig, agency_orig, complaint_orig] if c is not None]


In [ ]:
chunks_kept = []
rows_kept = 0
np.random.seed(SEED)

for chunk in pd.read_csv(CSV_PATH, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False):
    if rows_kept >= MAX_SAMPLED_ROWS:
        break
    sampled = chunk.sample(frac=SAMPLE_FRACTION, random_state=SEED)
    chunks_kept.append(sampled)
    rows_kept += len(sampled)

df_raw = pd.concat(chunks_kept, ignore_index=True)
if len(df_raw) > MAX_SAMPLED_ROWS:
    df_raw = df_raw.sample(n=MAX_SAMPLED_ROWS, random_state=SEED).reset_index(drop=True)

print("Sampled shape (spread across the full file, not just the earliest rows):", df_raw.shape)
df_raw.head(3)


## 3. Clean columns and build the `Descriptor -> Agency` dataset

Column detection already happened during the chunked load above (Section 2) -- `descriptor_orig` and
`agency_orig` point at the real column names in this file. We just rename them to consistent internal
names here.

In [ ]:
df_raw = df_raw.rename(columns={
    descriptor_orig: "descriptor_col",
    agency_orig: "agency_col",
    **({complaint_orig: "complaint_col"} if complaint_orig else {}),
})
descriptor_col = "descriptor_col"
agency_col = "agency_col"
complaint_col = "complaint_col" if complaint_orig else None


### Sanity check: does this dataset actually cover multiple agencies?

Some Kaggle uploads of "311 data" are scoped to a single department (e.g. HPD-only housing complaints).
If that's what you attached, `agency_col.nunique()` will come back as 1 or 2 -- there'd be nothing to
route between, and the whole classification task would be meaningless. Check this **before** running
any training below.

In [ ]:
n_agencies = df_raw[agency_col].nunique(dropna=True)
print(f"{n_agencies} unique agencies found in this dataset.")
print(df_raw[agency_col].value_counts().head(15))

if n_agencies < 5:
    print("\nWARNING: fewer than 5 distinct agencies detected. This dataset is likely scoped to one "
          "department (e.g. HPD-only). Go back to 'Add Data' and attach a full multi-agency 311 export "
          "instead, e.g. the 'new-york-city/ny-311-service-requests' dataset, or the official "
          "NYC Open Data 311 CSV export.")


In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df_raw[[descriptor_col, agency_col]].copy()
df.columns = ["text", "label"]
df = df.dropna(subset=["text", "label"])
df["text"] = df["text"].apply(preprocess_text)
df = df[df["text"].str.len() > 0]

print("Rows after cleaning:", len(df))
df["label"].value_counts().head(20)


### Cap to top-N agencies

Real 311 data has a long tail of rare agencies (some with only a handful of requests). We keep the
top `TOP_N_CLASSES` most frequent agencies and drop the rest, which keeps the task realistic (still
imbalanced) without a handful of near-empty classes making every model's minority-class metrics meaningless.

In [ ]:
TOP_N_CLASSES = 10

top_labels = df["label"].value_counts().nlargest(TOP_N_CLASSES).index.tolist()
df = df[df["label"].isin(top_labels)].reset_index(drop=True)

print(f"Kept {len(top_labels)} classes, {len(df)} rows total")
print(df["label"].value_counts())

plt.figure(figsize=(8, 5))
df["label"].value_counts().plot(kind="bar", color="steelblue")
plt.title("Class distribution (kept agencies)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("/kaggle/working/class_distribution.png")
plt.show()


In [ ]:
# Optional but recommended: balance via downsampling the majority classes so no single
# class dominates training. Comment this block out if you'd rather study raw imbalance directly.
BALANCE_CLASSES = True

if BALANCE_CLASSES:
    min_size = df["label"].value_counts().min()
    cap = max(min_size, 3000)  # don't shrink everything down to a tiny minority class; cap a floor
    parts = []
    for lbl, group in df.groupby("label"):
        n = min(len(group), cap)
        parts.append(group.sample(n=n, random_state=SEED))
    df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print("After capping majority classes:")
    print(df["label"].value_counts())


In [ ]:
label_encoder = LabelEncoder()
df["y"] = label_encoder.fit_transform(df["label"])
CLASS_NAMES = label_encoder.classes_.tolist()
NUM_LABELS = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["y"], random_state=SEED
)
print("Train:", train_df.shape, "Test:", test_df.shape)


## 4. Shared evaluation utilities

Every model (classical or transformer) reports through this same function, so the comparison table at the end is apples-to-apples.

In [ ]:
RESULTS = {}  # model_name -> dict of metrics, filled in as each model finishes

def evaluate_predictions(model_name, y_true, y_pred, train_time=None, inference_time=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    micro_f1 = f1_score(y_true, y_pred, average="micro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    report = classification_report(
        y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0
    )

    RESULTS[model_name] = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "per_class_report": report,
        "y_true": y_true,          # raw predictions kept for McNemar's test + error analysis
        "y_pred": y_pred,
        "train_time_sec": train_time,
        "inference_time_sec": inference_time,
    }

    print(f"\n===== {model_name} =====")
    print(f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Weighted-F1: {weighted_f1:.4f}")
    if train_time is not None:
        print(f"Train time: {train_time:.1f}s | Inference time ({len(y_pred)} samples): {inference_time:.2f}s "
              f"({1000*inference_time/len(y_pred):.3f} ms/sample)")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    safe_name = re.sub(r"[^a-zA-Z0-9]+", "_", model_name)
    plt.savefig(f"/kaggle/working/confusion_matrix_{safe_name}.png")
    plt.show()

    return RESULTS[model_name]


## 5. Baseline 1 — TF-IDF + Linear SVM

In [ ]:
import time

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["text"])
X_test_tfidf = tfidf.transform(test_df["text"])

t0 = time.time()
svm = LinearSVC(class_weight="balanced", random_state=SEED)
svm.fit(X_train_tfidf, train_df["y"])
svm_train_time = time.time() - t0

t0 = time.time()
svm_preds = svm.predict(X_test_tfidf)
svm_inference_time = time.time() - t0

evaluate_predictions("TF-IDF + LinearSVM", test_df["y"], svm_preds,
                      train_time=svm_train_time, inference_time=svm_inference_time)


## 6. Baseline 2 — TF-IDF + Multinomial Naive Bayes

In [ ]:
t0 = time.time()
nb = MultinomialNB()
nb.fit(X_train_tfidf, train_df["y"])
nb_train_time = time.time() - t0

t0 = time.time()
nb_preds = nb.predict(X_test_tfidf)
nb_inference_time = time.time() - t0

evaluate_predictions("TF-IDF + NaiveBayes", test_df["y"], nb_preds,
                      train_time=nb_train_time, inference_time=nb_inference_time)


## 7. Transformer models

Same train/test split, same labels, same evaluation function as the baselines above.

**Add more models:** append e.g. `"roberta-base"` to `TRANSFORMER_MODELS` if you have GPU-hours to spare
on Kaggle (each additional model roughly adds one full fine-tuning run's worth of time).

In [ ]:
TRANSFORMER_MODELS = {
    "BERT-base": "bert-base-uncased",
    "DeBERTa-v3-small": "microsoft/deberta-v3-small",
    # "RoBERTa-base": "roberta-base",   # uncomment to include a 3rd transformer
}

MAX_LEN = 96
EPOCHS = 3
BATCH_SIZE = 16 if DEVICE == "cuda" else 8

train_ds_raw = Dataset.from_pandas(train_df[["text", "y"]].rename(columns={"y": "label"}))
test_ds_raw = Dataset.from_pandas(test_df[["text", "y"]].rename(columns={"y": "label"}))


In [ ]:
def run_transformer(display_name, checkpoint):
    print(f"\n\n########## Training {display_name} ({checkpoint}) ##########")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

    train_ds = train_ds_raw.map(tokenize, batched=True)
    test_ds = test_ds_raw.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=NUM_LABELS)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    is_deberta = "deberta" in checkpoint.lower()

    # DeBERTa-v2/v3's relative-position/embedding layers produce gradients that the fp16
    # gradient scaler can't unscale correctly -- disable fp16 specifically for DeBERTa.
    use_fp16 = (DEVICE == "cuda") and not is_deberta

    # DeBERTa-v3 is prone to early-training collapse (predicting one class for everything)
    # with the default LR/epsilon used for BERT/RoBERTa -- needs a lower LR, warmup, and
    # smaller adam_epsilon to converge stably.
    lr = 1e-5 if is_deberta else 2e-5
    warmup_ratio = 0.1 if is_deberta else 0.0
    adam_eps = 1e-6 if is_deberta else 1e-8

    args = TrainingArguments(
        output_dir=f"/kaggle/working/{display_name.replace(' ', '_')}",
        eval_strategy="epoch",
        save_strategy="no",           # don't checkpoint every epoch -- saves Kaggle disk quota
        learning_rate=lr,
        warmup_ratio=warmup_ratio,
        adam_epsilon=adam_eps,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        logging_steps=100,
        report_to="none",
        fp16=use_fp16,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        data_collator=collator,
        processing_class=tokenizer,
    )

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    t0 = time.time()
    preds_output = trainer.predict(test_ds)
    inference_time = time.time() - t0
    y_pred = np.argmax(preds_output.predictions, axis=1)

    evaluate_predictions(display_name, test_df["y"].values, y_pred,
                          train_time=train_time, inference_time=inference_time)

    # free GPU memory before the next model
    del model, trainer
    torch.cuda.empty_cache()


for display_name, checkpoint in TRANSFORMER_MODELS.items():
    run_transformer(display_name, checkpoint)


## 8. Final comparison table + chart (drop straight into the paper)

In [ ]:
comparison_rows = []
for model_name, metrics in RESULTS.items():
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": round(metrics["accuracy"], 4),
        "Macro-F1": round(metrics["macro_f1"], 4),
        "Micro-F1": round(metrics["micro_f1"], 4),
        "Weighted-F1": round(metrics["weighted_f1"], 4),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("Macro-F1", ascending=False).reset_index(drop=True)
comparison_df.to_csv("/kaggle/working/model_comparison.csv", index=False)
comparison_df


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=comparison_df, x="Model", y="Macro-F1", palette="viridis")
plt.title("Macro-F1 by Model — Descriptor -> Agency Classification (NYC 311)")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("/kaggle/working/macro_f1_comparison.png")
plt.show()


## 9. Per-class error analysis for the best model

Look at which classes each model confuses most -- this is exactly the kind of concrete, defensible
finding a reviewer wants (e.g. "DeBERTa-v3 recovers minority-class recall better than TF-IDF+SVM on
Class X, but both struggle equally on semantically overlapping classes Y and Z").

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
print("Best model by Macro-F1:", best_model_name)

report_df = pd.DataFrame(RESULTS[best_model_name]["per_class_report"]).T
report_df = report_df.drop(index=["accuracy"], errors="ignore")
report_df


## 10. Efficiency comparison

Training time and inference latency matter for a *practical routing system* claim, not just accuracy --
worth its own table row in the paper, especially since the classical baselines are competitive on
accuracy too.

In [ ]:
efficiency_rows = []
for model_name, metrics in RESULTS.items():
    tt = metrics.get("train_time_sec")
    it = metrics.get("inference_time_sec")
    n_test = len(metrics["y_pred"])
    efficiency_rows.append({
        "Model": model_name,
        "Train Time (s)": round(tt, 2) if tt is not None else None,
        "Inference Time - total (s)": round(it, 2) if it is not None else None,
        "Inference Latency (ms/sample)": round(1000 * it / n_test, 4) if it is not None else None,
    })

efficiency_df = pd.DataFrame(efficiency_rows)
efficiency_df.to_csv("/kaggle/working/efficiency_comparison.csv", index=False)
efficiency_df


## 11. Statistical significance — McNemar's test

The accuracy gaps between BERT, SVM, and DeBERTa are small enough that a reviewer will ask whether
they're statistically meaningful or just noise. McNemar's test compares two classifiers on the *same*
test set by looking only at the examples where they disagree -- exactly the right test for this setup
(same train/test split, same evaluation set, paired predictions).

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def run_mcnemar(model_a, model_b):
    y_true = RESULTS[model_a]["y_true"]
    pred_a = RESULTS[model_a]["y_pred"]
    pred_b = RESULTS[model_b]["y_pred"]
    assert len(y_true) == len(pred_a) == len(pred_b), "Predictions must come from the identical test set"

    a_correct = (pred_a == y_true)
    b_correct = (pred_b == y_true)

    # contingency table: [[both correct, A correct & B wrong], [A wrong & B correct, both wrong]]
    both_correct = np.sum(a_correct & b_correct)
    a_only = np.sum(a_correct & ~b_correct)
    b_only = np.sum(~a_correct & b_correct)
    both_wrong = np.sum(~a_correct & ~b_correct)

    table = [[both_correct, a_only], [b_only, both_wrong]]
    result = mcnemar(table, exact=(a_only + b_only) < 25, correction=True)

    print(f"{model_a} vs {model_b}")
    print(f"  {model_a} correct only: {a_only} | {model_b} correct only: {b_only}")
    print(f"  statistic={result.statistic:.4f}, p-value={result.pvalue:.4g}"
          f"  -> {'SIGNIFICANT (p<0.05)' if result.pvalue < 0.05 else 'not significant (p>=0.05)'}")
    print()
    return result.pvalue

model_names = list(RESULTS.keys())
mcnemar_results = []
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        p = run_mcnemar(model_names[i], model_names[j])
        mcnemar_results.append({"Model A": model_names[i], "Model B": model_names[j], "p-value": p})

mcnemar_df = pd.DataFrame(mcnemar_results)
mcnemar_df.to_csv("/kaggle/working/mcnemar_results.csv", index=False)
mcnemar_df


## 12. Real misclassified examples

Pulling actual `Descriptor` text for the confused class pairs turns the confusion matrix into concrete,
quotable evidence for the paper's error-analysis subsection (e.g. showing *why* DOHMH gets confused with
DSNY/HPD is far more convincing than just citing the confusion matrix numbers).

In [ ]:
def show_misclassified_examples(model_name, true_class, predicted_class, n=5):
    metrics = RESULTS[model_name]
    y_true = metrics["y_true"]
    y_pred = metrics["y_pred"]

    true_idx = CLASS_NAMES.index(true_class)
    pred_idx = CLASS_NAMES.index(predicted_class)

    mask = (y_true == true_idx) & (y_pred == pred_idx)
    examples = test_df.loc[mask, "text"].head(n).tolist()

    print(f"\n[{model_name}] True={true_class}, Predicted={predicted_class}  ({mask.sum()} total cases)")
    for ex in examples:
        print(f"  - \"{ex}\"")
    return examples

# Adjust these pairs based on what your own confusion matrix actually shows as the biggest
# off-diagonal cells for the weaker model (commonly DeBERTa-v3-small in this setup).
weak_model = comparison_df.iloc[-1]["Model"]
print(f"Inspecting confusions for weakest model: {weak_model}\n")

report_df_weak = pd.DataFrame(RESULTS[weak_model]["per_class_report"]).T.drop(index=["accuracy"], errors="ignore")
worst_classes = report_df_weak.sort_values("f1-score").head(3).index.tolist()
print("Lowest F1-score classes for this model:", worst_classes)

cm_weak = confusion_matrix(RESULTS[weak_model]["y_true"], RESULTS[weak_model]["y_pred"])
cm_weak_df = pd.DataFrame(cm_weak, index=CLASS_NAMES, columns=CLASS_NAMES)
for true_class in worst_classes:
    row = cm_weak_df.loc[true_class].drop(true_class)
    top_confused_with = row.sort_values(ascending=False).index[0]
    if row.sort_values(ascending=False).iloc[0] > 0:
        show_misclassified_examples(weak_model, true_class, top_confused_with, n=5)


## 13. Data efficiency — Macro-F1 vs. training set size

Full-data accuracy ties between BERT and TF-IDF+SVM (McNemar's p=0.5, Section 11) don't tell the whole
story. Transformers typically pull ahead of classical models specifically in **low-data regimes**, since
their pretrained language knowledge substitutes for labeled examples the classical model doesn't have.

This sweep retrains all 4 models on shrinking fractions of the training set (100% down to 1%) and plots
Macro-F1 against training-set size on a log scale. It answers a more useful practical question than a
single full-data number: *"if you only had 5% as much labeled data, which model would you actually want?"*

**Runtime note:** transformer epochs are reduced to `EFFICIENCY_EPOCHS=2` here (vs. 3 in the main
experiment) to keep this sweep tractable across 6 fractions x 2 models -- expect roughly 25-30 extra
minutes on top of your earlier run.

In [ ]:
FRACTIONS = [1.0, 0.5, 0.25, 0.10, 0.05, 0.01]
EFFICIENCY_EPOCHS = 2

def subsample_train(frac):
    if frac >= 0.999:
        return train_df
    return (
        train_df.groupby("y", group_keys=False)
        .apply(lambda g: g.sample(n=max(1, int(round(len(g) * frac))), random_state=SEED))
        .reset_index(drop=True)
    )

def eval_classical_at_fraction(frac):
    sub = subsample_train(frac)
    vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
    Xtr = vec.fit_transform(sub["text"])
    Xte = vec.transform(test_df["text"])

    svm_m = LinearSVC(class_weight="balanced", random_state=SEED)
    svm_m.fit(Xtr, sub["y"])
    svm_f1 = f1_score(test_df["y"], svm_m.predict(Xte), average="macro")

    nb_m = MultinomialNB()
    nb_m.fit(Xtr, sub["y"])
    nb_f1 = f1_score(test_df["y"], nb_m.predict(Xte), average="macro")

    return svm_f1, nb_f1

def eval_transformer_at_fraction(checkpoint, is_deberta, frac):
    sub = subsample_train(frac)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

    tr_ds = Dataset.from_pandas(sub[["text", "y"]].rename(columns={"y": "label"})).map(tokenize, batched=True)
    te_ds = test_ds_raw.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=NUM_LABELS)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    use_fp16 = (DEVICE == "cuda") and not is_deberta
    lr = 1e-5 if is_deberta else 2e-5
    warmup_ratio = 0.1 if is_deberta else 0.0
    adam_eps = 1e-6 if is_deberta else 1e-8

    args = TrainingArguments(
        output_dir="/kaggle/working/eff_tmp",
        eval_strategy="no",
        save_strategy="no",
        learning_rate=lr,
        warmup_ratio=warmup_ratio,
        adam_epsilon=adam_eps,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EFFICIENCY_EPOCHS,
        weight_decay=0.01,
        logging_steps=200,
        report_to="none",
        fp16=use_fp16,
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=tr_ds, data_collator=collator, processing_class=tokenizer)
    trainer.train()
    preds = trainer.predict(te_ds)
    y_pred = np.argmax(preds.predictions, axis=1)
    f1 = f1_score(test_df["y"].values, y_pred, average="macro")

    del model, trainer
    torch.cuda.empty_cache()
    return f1


In [ ]:
efficiency_records = []

for frac in FRACTIONS:
    print(f"\n=== Fraction: {frac:.0%} of training data ===")
    svm_f1, nb_f1 = eval_classical_at_fraction(frac)
    bert_f1 = eval_transformer_at_fraction("bert-base-uncased", False, frac)
    deberta_f1 = eval_transformer_at_fraction("microsoft/deberta-v3-small", True, frac)

    for model_name, f1 in [
        ("TF-IDF + LinearSVM", svm_f1),
        ("TF-IDF + NaiveBayes", nb_f1),
        ("BERT-base", bert_f1),
        ("DeBERTa-v3-small", deberta_f1),
    ]:
        efficiency_records.append({"Fraction": frac, "Model": model_name, "Macro-F1": f1})

    print(f"  SVM={svm_f1:.4f}  NB={nb_f1:.4f}  BERT={bert_f1:.4f}  DeBERTa={deberta_f1:.4f}")

data_efficiency_df = pd.DataFrame(efficiency_records)
data_efficiency_df.to_csv("/kaggle/working/data_efficiency.csv", index=False)
data_efficiency_df


In [ ]:
plt.figure(figsize=(9, 6))
for model_name, group in data_efficiency_df.groupby("Model"):
    group = group.sort_values("Fraction")
    plt.plot(group["Fraction"] * 100, group["Macro-F1"], marker="o", label=model_name)

plt.xscale("log")
plt.xlabel("Training data used (%, log scale)")
plt.ylabel("Macro-F1")
plt.title("Data Efficiency: Macro-F1 vs. Training Set Size")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/data_efficiency_curve.png")
plt.show()


## 14. Confirmatory check — is the 1% collapse a step-count artifact?

At 1% of training data (~250-300 examples) with only 2 epochs, both transformers collapsed
(BERT to 0.19, DeBERTa to 0.02) while the classical baselines stayed above 0.89. The likely explanation:
at this data size, 2 epochs is only ~30-40 total gradient update steps -- not enough for a randomly
initialized classification head on a 100M+ parameter model to move off its random initialization.

This check retrains BERT and DeBERTa **on the exact same 1% split** at increasing epoch counts
(2, 5, 10, 15, 20) and re-evaluates each time. If Macro-F1 climbs back toward the classical baselines'
~0.90 as epochs increase, that confirms it's a step-count issue, not a genuine low-data weakness of
transformers. This is fast -- ~250-300 examples means even 20 epochs is well under a thousand total
optimizer steps per run.

In [ ]:
def eval_transformer_at_fraction_custom_epochs(checkpoint, is_deberta, frac, epochs):
    sub = subsample_train(frac)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

    tr_ds = Dataset.from_pandas(sub[["text", "y"]].rename(columns={"y": "label"})).map(tokenize, batched=True)
    te_ds = test_ds_raw.map(tokenize, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=NUM_LABELS)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    use_fp16 = (DEVICE == "cuda") and not is_deberta
    lr = 1e-5 if is_deberta else 2e-5
    warmup_ratio = 0.1 if is_deberta else 0.0
    adam_eps = 1e-6 if is_deberta else 1e-8

    args = TrainingArguments(
        output_dir="/kaggle/working/confirm_tmp",
        eval_strategy="no",
        save_strategy="no",
        learning_rate=lr,
        warmup_ratio=warmup_ratio,
        adam_epsilon=adam_eps,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
        fp16=use_fp16,
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=tr_ds, data_collator=collator, processing_class=tokenizer)
    trainer.train()
    preds = trainer.predict(te_ds)
    y_pred = np.argmax(preds.predictions, axis=1)
    f1 = f1_score(test_df["y"].values, y_pred, average="macro")

    del model, trainer
    torch.cuda.empty_cache()
    return f1


In [ ]:
EPOCH_SWEEP = [2, 5, 10, 15, 20]
CONFIRM_FRACTION = 0.01  # same 1% split that collapsed

confirm_records = []
for epochs_try in EPOCH_SWEEP:
    bert_f1 = eval_transformer_at_fraction_custom_epochs("bert-base-uncased", False, CONFIRM_FRACTION, epochs_try)
    deberta_f1 = eval_transformer_at_fraction_custom_epochs("microsoft/deberta-v3-small", True, CONFIRM_FRACTION, epochs_try)

    confirm_records.append({"Epochs": epochs_try, "Model": "BERT-base", "Macro-F1": bert_f1})
    confirm_records.append({"Epochs": epochs_try, "Model": "DeBERTa-v3-small", "Macro-F1": deberta_f1})

    print(f"epochs={epochs_try:2d}:  BERT-base={bert_f1:.4f}   DeBERTa-v3-small={deberta_f1:.4f}")

confirm_df = pd.DataFrame(confirm_records)
confirm_df.to_csv("/kaggle/working/low_data_epoch_confirmation.csv", index=False)
confirm_df


In [ ]:
# Reference lines: classical baselines' Macro-F1 at the same 1% fraction, from Section 13
classical_at_1pct = data_efficiency_df[
    (data_efficiency_df["Fraction"] == CONFIRM_FRACTION) &
    (data_efficiency_df["Model"].isin(["TF-IDF + LinearSVM", "TF-IDF + NaiveBayes"]))
]

plt.figure(figsize=(9, 6))
for model_name, group in confirm_df.groupby("Model"):
    group = group.sort_values("Epochs")
    plt.plot(group["Epochs"], group["Macro-F1"], marker="o", label=model_name)

for _, row in classical_at_1pct.iterrows():
    plt.axhline(row["Macro-F1"], linestyle="--", alpha=0.5,
                label=f'{row["Model"]} (classical, fixed)')

plt.xlabel("Training epochs (on the same 1% split)")
plt.ylabel("Macro-F1")
plt.title("Does More Training Recover the 1%-Data Collapse?")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/low_data_epoch_confirmation.png")
plt.show()


## 15. Where this feeds into the paper

- **Results section**: use `comparison_df` (Section 8) as your main results table, and the confusion
  matrices / per-class report (Sections 4, 9) for the error-analysis subsection.
- **Data-efficiency subsection**: `data_efficiency_df` / the curve (Section 13) is your answer to "but
  the accuracy is basically the same" -- show where BERT/DeBERTa pull ahead of the classical baselines as
  training data shrinks, which justifies transformers' value even without a full-data accuracy edge.
- **Efficiency subsection**: `efficiency_df` (Section 10) supports a "practical deployability" angle --
  classical baselines training in seconds vs. transformers taking minutes is a real, citable tradeoff.
- **Significance**: `mcnemar_df` (Section 11) tells you which pairwise differences in the results table
  are actually statistically significant vs. within noise -- report this alongside the raw F1 numbers
  rather than treating small gaps as proven differences.
- **Discussion**: contrast where transformers win vs. where TF-IDF+SVM is competitive (short descriptor
  text often favors classical models more than long-document tasks do -- worth discussing explicitly if
  that's what you observe here), and use the concrete misclassified examples (Section 12) as illustrative
  quotes for *why* specific classes get confused.
- **Applied case study section**: the best-performing model from `comparison_df` is the one to plug into
  the ticketing / SLA / notification pipeline from the original GuardianX notebook, positioned as the
  deployed system built on top of these benchmarking results -- not as a separate contribution.
